# Avaliação — APIs de energia renovável e aprendizado de máquina

## Tarefas 1 e 2 resolvidas

Este notebook contém as duas atividades propostas:

- **Tarefa 1:** classificação da fonte de empreendimentos da ANEEL;
- **Tarefa 2:** regressão para estimativa da radiação solar em Petrolina (PE).

As duas tarefas possuem análise exploratória, preparação dos dados, treinamento de três modelos, métricas, visualizações e interpretação dos resultados.


## 1. Consulta e preparação dos dados da ANEEL

A API pública do SIGA é consultada para as cinco categorias previstas no exercício: `UFV`, `EOL`, `UHE`, `PCH` e `CGH`. As categorias `UHE`, `PCH` e `CGH` são agrupadas como **Hidráulica**.


In [ ]:
URL_ANEEL = "https://dadosabertos.aneel.gov.br/api/3/action/datastore_search"
ID_RECURSO = "11ec447d-698d-4ab8-977f-b424d5deee6a"

SIGLAS = ["UFV", "EOL", "UHE", "PCH", "CGH"]

registros_aneel = []

for sigla in SIGLAS:
    parametros = {
        "resource_id": ID_RECURSO,
        "filters": json.dumps({"SigTipoGeracao": sigla}),
        "limit": 1200,
    }

    resposta = consultar_api(URL_ANEEL, parametros)

    if not resposta.get("success"):
        raise ValueError(f"Consulta da ANEEL falhou para {sigla}")

    linhas = resposta["result"]["records"]
    print(f"{sigla}: {len(linhas)} linhas recebidas")
    registros_aneel.extend(linhas)

print("Total de linhas recebidas:", len(registros_aneel))

In [ ]:
def numero(valor):
    texto = str(valor).strip()

    if "," in texto:
        texto = texto.replace(".", "").replace(",", ".")

    try:
        return float(texto)
    except (ValueError, TypeError):
        return None


fontes = {
    "UFV": "Solar",
    "EOL": "Eólica",
    "UHE": "Hidráulica",
    "PCH": "Hidráulica",
    "CGH": "Hidráulica"
}

linhas_classificacao = []

for registro in registros_aneel:
    potencia = numero(registro.get("MdaPotenciaOutorgadaKw"))
    latitude = numero(registro.get("NumCoordNEmpreendimento"))
    longitude = numero(registro.get("NumCoordEEmpreendimento"))
    fonte = fontes.get(registro.get("SigTipoGeracao"))

    if fonte and all(
        valor is not None
        for valor in (potencia, latitude, longitude)
    ):
        linhas_classificacao.append([
            potencia,
            latitude,
            longitude,
            fonte
        ])

dados_classificacao = pd.DataFrame(
    linhas_classificacao,
    columns=["potencia_kw", "latitude", "longitude", "fonte"]
)

print("Linhas válidas:", len(dados_classificacao))
display(dados_classificacao.head())

In [ ]:
dados_classificacao.to_csv(
    "aneel_classificacao_orange.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV salvo como: aneel_classificacao_orange.csv")

## 2. Análise exploratória

Primeiro verificamos o tamanho do conjunto, os tipos, os valores ausentes, a distribuição das classes e as estatísticas das três entradas.


In [ ]:
print("Dimensões:", dados_classificacao.shape)

print("\nTipos das colunas:")
print(dados_classificacao.dtypes)

print("\nValores ausentes:")
print(dados_classificacao.isna().sum())

print("\nQuantidade por classe:")
display(dados_classificacao["fonte"].value_counts())

print("\nPercentual por classe:")
display(
    dados_classificacao["fonte"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nEstatísticas das entradas:")
display(dados_classificacao.describe())

In [ ]:
plt.figure(figsize=(8, 5))
dados_classificacao["fonte"].value_counts().plot(kind="bar")
plt.title("Quantidade de empreendimentos por fonte")
plt.xlabel("Fonte")
plt.ylabel("Quantidade")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()

plt.figure(figsize=(8, 6))
for classe in dados_classificacao["fonte"].unique():
    grupo = dados_classificacao[dados_classificacao["fonte"] == classe]
    plt.scatter(
        grupo["longitude"],
        grupo["latitude"],
        alpha=0.35,
        label=classe
    )

plt.title("Distribuição geográfica dos empreendimentos")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. Definição de X e y e divisão estratificada

As entradas são somente:

- `potencia_kw`
- `latitude`
- `longitude`

A variável `fonte` é o alvo.

A divisão é estratificada para preservar aproximadamente a proporção das três classes tanto no treino quanto no teste.


In [ ]:
features_classificacao = [
    "potencia_kw",
    "latitude",
    "longitude"
]

X_class = dados_classificacao[features_classificacao]
y_class = dados_classificacao["fonte"]

X_treino_class, X_teste_class, y_treino_class, y_teste_class = train_test_split(
    X_class,
    y_class,
    test_size=0.20,
    random_state=42,
    stratify=y_class
)

print(f"Treino: {len(X_treino_class)} registros")
print(f"Teste: {len(X_teste_class)} registros")

print("\nDistribuição no treino:")
display(y_treino_class.value_counts(normalize=True).mul(100).round(2))

print("\nDistribuição no teste:")
display(y_teste_class.value_counts(normalize=True).mul(100).round(2))

## 4. Treinamento dos três classificadores

Foram escolhidos três algoritmos diferentes:

1. **KNN** — classifica observações com base na proximidade de seus vizinhos;
2. **Regressão Logística** — modelo linear para classificação multiclasses;
3. **Random Forest** — conjunto de árvores de decisão capaz de representar relações não lineares.

Para KNN e Regressão Logística, a padronização é feita dentro de um `Pipeline`. Assim, o `StandardScaler` aprende os parâmetros **somente com o conjunto de treino**, evitando vazamento de dados.


In [ ]:
classificadores = {
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", KNeighborsClassifier(n_neighbors=7))
    ]),

    "Regressão Logística": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    )
}

resultados_classificacao = []
previsoes_classificacao = {}

for nome, modelo in classificadores.items():
    modelo.fit(X_treino_class, y_treino_class)
    y_pred = modelo.predict(X_teste_class)
    previsoes_classificacao[nome] = y_pred

    resultados_classificacao.append({
        "Modelo": nome,
        "Accuracy": accuracy_score(y_teste_class, y_pred),
        "Precision (macro)": precision_score(
            y_teste_class, y_pred, average="macro", zero_division=0
        ),
        "Recall (macro)": recall_score(
            y_teste_class, y_pred, average="macro", zero_division=0
        ),
        "F1 (macro)": f1_score(
            y_teste_class, y_pred, average="macro", zero_division=0
        )
    })

comparacao_classificacao = pd.DataFrame(resultados_classificacao).sort_values(
    "F1 (macro)", ascending=False
).reset_index(drop=True)

display(
    comparacao_classificacao.style.format({
        "Accuracy": "{:.4f}",
        "Precision (macro)": "{:.4f}",
        "Recall (macro)": "{:.4f}",
        "F1 (macro)": "{:.4f}"
    })
)

## 5. Matrizes de confusão

A matriz de confusão mostra quais classes foram corretamente classificadas e quais foram confundidas entre si.

- Linhas: classe real
- Colunas: classe prevista


In [ ]:
classes = ["Solar", "Eólica", "Hidráulica"]

for nome, y_pred in previsoes_classificacao.items():
    matriz = confusion_matrix(
        y_teste_class,
        y_pred,
        labels=classes
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=matriz,
        display_labels=classes
    )

    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Matriz de confusão — {nome}")
    plt.tight_layout()
    plt.show()

## 6. Relatórios detalhados por classe

Além das métricas macro, o relatório abaixo mostra Precision, Recall e F1 individualmente para Solar, Eólica e Hidráulica.


In [ ]:
for nome, y_pred in previsoes_classificacao.items():
    print("=" * 70)
    print(f"Modelo: {nome}")
    print("=" * 70)
    print(
        classification_report(
            y_teste_class,
            y_pred,
            labels=classes,
            target_names=classes,
            zero_division=0
        )
    )

## 7. Interpretação da classificação

As métricas `Precision`, `Recall` e `F1` foram calculadas com **média macro**, dando o mesmo peso às três classes independentemente de sua quantidade de exemplos.

Na interpretação dos resultados, deve-se observar principalmente quais classes apresentam maior número de erros na matriz de confusão. A confusão entre classes pode ocorrer porque potência e localização, isoladamente, não carregam todas as características necessárias para identificar a tecnologia de geração.

A potência de um empreendimento pode variar bastante dentro de uma mesma fonte, enquanto regiões geográficas podem apresentar diferentes tipos de geração. Portanto, mesmo um modelo com bom desempenho no conjunto de teste pode apresentar limitações quando aplicado a empreendimentos com características diferentes das observadas nos dados de treinamento.

Também é importante destacar que não foram utilizadas informações como nome do empreendimento, código CEG, sigla do tipo de geração ou descrição da fonte, pois essas informações poderiam revelar diretamente a classe que o modelo deveria aprender a prever.


## 8. Conclusão — Tarefa 1

Nesta tarefa, foi desenvolvido um problema de classificação multiclasses para identificar a fonte de geração de empreendimentos como Solar, Eólica ou Hidráulica. Foram utilizadas somente a potência outorgada e as coordenadas geográficas como variáveis de entrada.

Os três classificadores — KNN, Regressão Logística e Random Forest — foram treinados utilizando a mesma divisão estratificada de treino e teste. A comparação foi realizada por Accuracy, Precision macro, Recall macro e F1 macro, além das matrizes de confusão.

Os resultados mostram como diferentes algoritmos podem apresentar comportamentos distintos diante das mesmas variáveis. A análise das matrizes de confusão também permite identificar quais fontes são mais frequentemente confundidas. Apesar dos resultados obtidos, potência e localização não são suficientes para representar todas as características de um empreendimento de geração, portanto a classificação deve ser entendida dentro das limitações do conjunto de dados utilizado.


# Exercício de Regressão — Radiação Solar em Petrolina (PE)

## Objetivo

Estimar a **radiação solar horizontal média por hora (`radiacao_w_m2`)** em Petrolina (PE) a partir das condições meteorológicas e da hora local.

A solução segue as instruções da atividade:

- dados históricos do Open-Meteo;
- período de 01/04/2025 a 30/06/2025;
- somente horários entre 7h e 17h;
- divisão temporal de aproximadamente 80% para treino e 20% para teste;
- sem embaralhar os registros;
- três algoritmos de regressão;
- comparação por **MAE, MSE e R²**;
- visualizações e interpretação dos resultados.


In [ ]:
import json
import urllib.parse
import urllib.request
import csv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Bibliotecas carregadas com sucesso.")

# Classificação
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)


In [ ]:
def consultar_api(endereco, parametros):
    url = endereco + "?" + urllib.parse.urlencode(parametros)
    try:
        with urllib.request.urlopen(url, timeout=100) as resposta:
            return json.load(resposta)
    except Exception as erro:
        raise RuntimeError(f"Não foi possível consultar a API: {erro}") from erro

## 1. Consulta à API Open-Meteo

As coordenadas, o período e as variáveis abaixo são os mesmos definidos no enunciado.

In [ ]:
URL_METEO = "https://archive-api.open-meteo.com/v1/archive"

parametros_meteo = {
    "latitude": -9.39,
    "longitude": -40.50,
    "start_date": "2025-04-01",
    "end_date": "2025-06-30",
    "timezone": "America/Recife",
    "hourly": ",".join([
        "temperature_2m",
        "relative_humidity_2m",
        "cloud_cover",
        "wind_speed_10m",
        "shortwave_radiation"
    ]),
}

resposta_meteo = consultar_api(URL_METEO, parametros_meteo)

if "hourly" not in resposta_meteo:
    raise ValueError("A resposta da API não contém a seção 'hourly'.")

horas_api = resposta_meteo["hourly"]

print("Campos recebidos:")
print(list(horas_api))
print("\nUnidades:")
print(resposta_meteo.get("hourly_units", {}))
print("\nQuantidade total de horários:", len(horas_api["time"]))

## 2. Preparação dos dados

As listas retornadas pela API são alinhadas pelo índice. Mantemos apenas registros entre 7h e 17h e removemos linhas com valores ausentes.

A variável-alvo é `radiacao_w_m2`. Ela **não será utilizada como entrada**.

In [ ]:
nomes_api = [
    "temperature_2m",
    "relative_humidity_2m",
    "cloud_cover",
    "wind_speed_10m",
    "shortwave_radiation"
]

linhas_regressao = []

for indice, data_hora in enumerate(horas_api["time"]):
    hora = int(data_hora[11:13])
    valores = [horas_api[nome][indice] for nome in nomes_api]

    if 7 <= hora <= 17 and all(valor is not None for valor in valores):
        linhas_regressao.append([
            data_hora,
            *valores[:4],
            hora,
            valores[4]
        ])

if len(linhas_regressao) < 100:
    raise ValueError("Poucos horários válidos. Confira a resposta do Open-Meteo.")

dados = pd.DataFrame(
    linhas_regressao,
    columns=[
        "data_hora",
        "temperatura_c",
        "umidade_pct",
        "nuvens_pct",
        "vento_kmh",
        "hora",
        "radiacao_w_m2"
    ]
)

dados["data_hora"] = pd.to_datetime(dados["data_hora"])
dados = dados.sort_values("data_hora").reset_index(drop=True)

print("Quantidade de registros válidos:", len(dados))
dados.head()

In [ ]:
# Salva o CSV para permitir a reprodução da atividade.
dados.to_csv(
    "meteo_regressao_orange.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV salvo como: meteo_regressao_orange.csv")

## 3. Análise inicial

As cinco entradas serão:

- `temperatura_c`
- `umidade_pct`
- `nuvens_pct`
- `vento_kmh`
- `hora`

O alvo será:

- `radiacao_w_m2`

A coluna `data_hora` é usada para manter a ordem temporal e não entra diretamente no modelo.

In [ ]:
print("Informações do conjunto:")
dados.info()

print("\nValores ausentes:")
print(dados.isna().sum())

print("\nEstatísticas descritivas:")
display(dados.describe())

print("\nPrimeiras linhas:")
display(dados.head(10))

In [ ]:
# Distribuição da radiação solar
plt.figure(figsize=(10, 5))
plt.hist(dados["radiacao_w_m2"], bins=30)
plt.title("Distribuição da radiação solar")
plt.xlabel("Radiação (W/m²)")
plt.ylabel("Frequência")
plt.grid(alpha=0.3)
plt.show()

# Relação entre hora e radiação
plt.figure(figsize=(10, 5))
plt.scatter(dados["hora"], dados["radiacao_w_m2"], alpha=0.25)
plt.title("Radiação solar por hora do dia")
plt.xlabel("Hora local")
plt.ylabel("Radiação (W/m²)")
plt.grid(alpha=0.3)
plt.show()

## 4. Separação temporal entre treino e teste

Como os dados são horários e possuem ordem temporal, não devemos embaralhá-los.

Usaremos:

- **80% das primeiras observações** → treino;
- **20% das últimas observações** → teste.

Essa abordagem simula uma situação em que o modelo aprende com dados anteriores e depois é avaliado em dados posteriores.

In [ ]:
features = [
    "temperatura_c",
    "umidade_pct",
    "nuvens_pct",
    "vento_kmh",
    "hora"
]

alvo = "radiacao_w_m2"

X = dados[features].copy()
y = dados[alvo].copy()

ponto_corte = int(len(dados) * 0.80)

X_treino = X.iloc[:ponto_corte].copy()
X_teste = X.iloc[ponto_corte:].copy()

y_treino = y.iloc[:ponto_corte].copy()
y_teste = y.iloc[ponto_corte:].copy()

print(f"Total de registros: {len(dados)}")
print(f"Treino: {len(X_treino)} ({len(X_treino)/len(dados):.1%})")
print(f"Teste:  {len(X_teste)} ({len(X_teste)/len(dados):.1%})")
print(f"Última data/hora do treino: {dados['data_hora'].iloc[ponto_corte-1]}")
print(f"Primeira data/hora do teste: {dados['data_hora'].iloc[ponto_corte]}")

## 5. Treinamento dos três modelos

Serão comparados:

1. **Regressão Linear** — modelo simples e interpretável, adequado como baseline;
2. **Árvore de Decisão** — consegue representar relações não lineares;
3. **Random Forest** — conjunto de várias árvores, geralmente mais robusto a relações complexas.

Todos recebem exatamente a mesma divisão treino/teste.

In [ ]:
modelos = {
    "Regressão Linear": LinearRegression(),
    "Árvore de Decisão": DecisionTreeRegressor(
        max_depth=8,
        random_state=42
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    )
}

resultados = []
previsoes = {}

for nome, modelo in modelos.items():
    modelo.fit(X_treino, y_treino)
    y_pred = modelo.predict(X_teste)

    previsoes[nome] = y_pred

    mae = mean_absolute_error(y_teste, y_pred)
    mse = mean_squared_error(y_teste, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_teste, y_pred)

    resultados.append({
        "Modelo": nome,
        "MAE (W/m²)": mae,
        "MSE ((W/m²)²)": mse,
        "RMSE (W/m²)": rmse,
        "R²": r2
    })

comparacao = pd.DataFrame(resultados).sort_values(
    "MAE (W/m²)"
).reset_index(drop=True)

comparacao

## 6. Comparação dos modelos

**MAE** representa o erro absoluto médio em W/m². Quanto menor, melhor.

**MSE** penaliza mais fortemente erros grandes. Quanto menor, melhor.

**R²** indica quanto da variação do alvo é explicada pelo modelo. Valores mais próximos de 1 indicam maior capacidade explicativa no conjunto de teste.

In [ ]:
display(comparacao.style.format({
    "MAE (W/m²)": "{:.2f}",
    "MSE ((W/m²)²)": "{:.2f}",
    "RMSE (W/m²)": "{:.2f}",
    "R²": "{:.4f}"
}))

In [ ]:
# Comparação visual das métricas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(comparacao["Modelo"], comparacao["MAE (W/m²)"])
axes[0].set_title("MAE por modelo")
axes[0].set_ylabel("MAE (W/m²)")
axes[0].tick_params(axis="x", rotation=20)
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(comparacao["Modelo"], comparacao["R²"])
axes[1].set_title("R² por modelo")
axes[1].set_ylabel("R²")
axes[1].tick_params(axis="x", rotation=20)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Valores reais × previstos

A seguir, o gráfico usa o modelo **Random Forest** como exemplo. A linha de referência representa a situação ideal em que o valor previsto seria exatamente igual ao valor observado.

In [ ]:
modelo_grafico = "Random Forest"
y_pred_grafico = previsoes[modelo_grafico]

plt.figure(figsize=(8, 8))
plt.scatter(y_teste, y_pred_grafico, alpha=0.35)

limite_min = min(y_teste.min(), y_pred_grafico.min())
limite_max = max(y_teste.max(), y_pred_grafico.max())

plt.plot(
    [limite_min, limite_max],
    [limite_min, limite_max],
    linestyle="--"
)

plt.title(f"Valores reais × previstos — {modelo_grafico}")
plt.xlabel("Radiação real (W/m²)")
plt.ylabel("Radiação prevista (W/m²)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Evolução temporal: valores reais e previstos no conjunto de teste
plt.figure(figsize=(14, 5))

plt.plot(
    dados["data_hora"].iloc[ponto_corte:],
    y_teste.values,
    label="Real"
)

plt.plot(
    dados["data_hora"].iloc[ponto_corte:],
    y_pred_grafico,
    label="Previsto",
    alpha=0.8
)

plt.title(f"Radiação real × prevista ao longo do tempo — {modelo_grafico}")
plt.xlabel("Data e hora")
plt.ylabel("Radiação (W/m²)")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 8. Interpretação dos resultados

Os resultados numéricos da tabela acima são calculados automaticamente quando o notebook é executado, pois os dados são obtidos diretamente da API histórica do Open-Meteo.

A interpretação deve considerar:

- **MAE:** mede o erro médio em termos diretamente interpretáveis de W/m²;
- **MSE:** destaca mais os erros muito grandes;
- **R²:** permite avaliar a proporção da variabilidade da radiação explicada pelo modelo;
- a comparação deve usar o **mesmo conjunto de teste** para os três algoritmos.

### Sobre a hora

A variável `hora` é importante porque a radiação solar possui forte comportamento diário. Portanto, duas condições meteorológicas semelhantes podem estar associadas a níveis diferentes de radiação dependendo do horário.

### Sobre a limitação do modelo

A previsão de `radiacao_w_m2` **não equivale à previsão de energia elétrica produzida por um sistema fotovoltaico**. Para estimar geração seriam necessários outros fatores, como potência instalada, área e características dos módulos, eficiência, orientação e inclinação, perdas do sistema, temperatura dos módulos e demais condições operacionais.

Além disso, os dados do Open-Meteo usados na atividade são históricos estimados por modelos/reanálise; eles não representam necessariamente a leitura de um sensor específico instalado em Petrolina.

## 9. Conclusão

Foram treinados e comparados três algoritmos de regressão — **Regressão Linear, Árvore de Decisão e Random Forest** — utilizando a mesma divisão temporal de treino e teste.

A escolha do modelo para uma aplicação real deve considerar conjuntamente os erros obtidos, a estabilidade das previsões e as características do problema. Neste exercício, a tabela de métricas e os gráficos fornecem a base objetiva para essa comparação.

> **Importante:** execute todas as células em ordem para gerar os valores reais das métricas a partir dos dados da API.